####  Building LLM application using LCEL
* LangChain Expression Language
* LCEL (LangChain Expression Language) is a declarative way to link AI building blocks like prompts, models, and parsers using a pipe symbol (|).
* In this app we will do text translation from english to another language using LLM.
* This is relatively simple LLM application.

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
# GROQ_API_KEY=os.getenv("GROQ_API_KEY")

os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [18]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

model=ChatGroq(model="openai/gpt-oss-safeguard-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15'}}, profile={'name': 'Safety GPT OSS 20B', 'release_date': '2025-03-05', 'last_updated': '2025-03-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023EA54B1D20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023EA54B3610>, model_name='openai/gpt-oss-safeguard-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [19]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to Hindi"),
    HumanMessage(content="Hello, How are you?")
]
result=model.invoke(messages)

In [22]:
result

AIMessage(content='नमस्ते, आप कैसे हैं?', additional_kwargs={'reasoning_content': 'We need to translate "Hello, How are you?" from English to Hindi. Simple translation: "नमस्ते, आप कैसे हैं?" or "हैलो, आप कैसे हैं?" Use Hindi. Probably "नमस्ते, आप कैसे हैं?" Provide that.'}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 87, 'total_tokens': 162, 'completion_time': 0.079035865, 'completion_tokens_details': {'reasoning_tokens': 57}, 'prompt_time': 0.004826031, 'prompt_tokens_details': None, 'queue_time': 0.048846189, 'total_time': 0.083861896}, 'model_name': 'openai/gpt-oss-safeguard-20b', 'system_fingerprint': 'fp_b5ae46a825', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0247b-2b09-7d82-9434-ead980626155-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 75, 'total_tokens': 162, 'output_token_details': {'reasoning': 57}})

In [23]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parsed_result=parser.invoke(result)

In [24]:
parsed_result

'नमस्ते, आप कैसे हैं?'

In [25]:
### using LCEL ---> chaining the components

chain=model|parser
chain.invoke(messages)

'नमस्ते, आप कैसे हैं?'

In [38]:
### Prompt Template
from langchain_core.prompts import ChatPromptTemplate
generic_template="Translate the following text into {language}:"
prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)

In [39]:
message=prompt.invoke({"language":"Hindi","text":"where were you yeasterday?"})

In [40]:
message.to_messages()

[SystemMessage(content='Translate the following text into Hindi:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='where were you yeasterday?', additional_kwargs={}, response_metadata={})]

In [41]:
chain=prompt|model|parser
chain.invoke({"language":"Hindi","text":"where were you yeasterday?"})


'आप कल कहाँ थे?'